# 💻 Laptop Price Prediction
Beginner Machine Learning Project

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
# Change the path if your CSV has a different name
df = pd.read_csv('/kaggle/input/laptop-price/Laptop_price.csv')
df.head()

In [ ]:
print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

In [ ]:
print("Missing values:\n", df.isnull().sum())
print("\nDuplicates:", df.duplicated().sum())

In [ ]:
# Target column
target = 'Price'

X = df.drop(columns=[target])
y = df[target]

print("Features:", X.shape)
print("Target:", y.shape)

In [ ]:
# Numerical and categorical columns
numerical_columns = X.select_dtypes(include=np.number).columns.tolist()
categorical_columns = X.select_dtypes(include='object').columns.tolist()

print("Numerical:", numerical_columns)
print("Categorical:", categorical_columns)

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(y, bins=30)
plt.xlabel("Laptop Price")
plt.ylabel("Frequency")
plt.title("Laptop Price Distribution")
plt.show()

In [ ]:
if 'Company' in df.columns:
    plt.figure(figsize=(12,5))
    sns.boxplot(data=df, x='Company', y='Price')
    plt.xticks(rotation=45)
    plt.title("Price by Company")
    plt.show()

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df.select_dtypes(include=np.number).corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()

In [ ]:
# Fill missing values
for col in numerical_columns:
    X[col] = X[col].fillna(X[col].median())

for col in categorical_columns:
    X[col] = X[col].fillna(X[col].mode()[0])

print("Missing values:", X.isnull().sum().sum())

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)

In [ ]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

X_train_encoded = encoder.fit_transform(X_train[categorical_columns])
X_test_encoded = encoder.transform(X_test[categorical_columns])

print(X_train_encoded.shape, X_test_encoded.shape)

In [ ]:
X_train_num = X_train[numerical_columns].values
X_test_num = X_test[numerical_columns].values

X_train_final = np.hstack([X_train_num, X_train_encoded])
X_test_final = np.hstack([X_test_num, X_test_encoded])

print("Final training:", X_train_final.shape)
print("Final testing:", X_test_final.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_final)
X_test_scaled = scaler.transform(X_test_final)

print("Scaling completed successfully!")

## Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model = LinearRegression()
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

In [ ]:
plt.figure(figsize=(8,6))
plt.scatter(y_test, y_pred, alpha=0.6)
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted Laptop Price")
plt.show()

## Ridge Regression

In [ ]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)

ridge_pred = ridge.predict(X_test_scaled)

ridge_mae = mean_absolute_error(y_test, ridge_pred)
ridge_rmse = np.sqrt(mean_squared_error(y_test, ridge_pred))
ridge_r2 = r2_score(y_test, ridge_pred)

print("MAE:", ridge_mae)
print("RMSE:", ridge_rmse)
print("R²:", ridge_r2)

## Lasso Regression

In [ ]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.01, max_iter=10000)
lasso.fit(X_train_scaled, y_train)

lasso_pred = lasso.predict(X_test_scaled)

lasso_mae = mean_absolute_error(y_test, lasso_pred)
lasso_rmse = np.sqrt(mean_squared_error(y_test, lasso_pred))
lasso_r2 = r2_score(y_test, lasso_pred)

print("MAE:", lasso_mae)
print("RMSE:", lasso_rmse)
print("R²:", lasso_r2)

## Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Ridge', 'Lasso'],
    'MAE': [mae, ridge_mae, lasso_mae],
    'RMSE': [rmse, ridge_rmse, lasso_rmse],
    'R2': [r2, ridge_r2, lasso_r2]
})

results

In [ ]:
results.to_csv('laptop_model_results.csv', index=False)
print("Results saved successfully!")

## New Laptop Prediction

In [ ]:
# Example input - change values according to your dataset
new_laptop = pd.DataFrame({
    'Company': ['Dell'],
    'TypeName': ['Notebook'],
    'Inches': [15.6],
    'ScreenResolution': ['Full HD 1920x1080'],
    'Cpu': ['Intel Core i5'],
    'Ram': ['8GB'],
    'Memory': ['512GB SSD'],
    'Gpu': ['Intel'],
    'OpSys': ['Windows 10'],
    'Weight': ['1.8kg']
})

In [ ]:
new_num = new_laptop[numerical_columns].values
new_cat = encoder.transform(new_laptop[categorical_columns])

new_final = np.hstack([new_num, new_cat])
new_scaled = scaler.transform(new_final)

prediction = model.predict(new_scaled)

print(f"Predicted Laptop Price: ₹{prediction[0]:,.2f}")

## Conclusion

The dataset was analyzed and preprocessed using missing-value handling, One Hot Encoding and Standard Scaling. Linear, Ridge and Lasso Regression models were trained and evaluated using MAE, RMSE and R². The trained model can also predict the price of a new laptop.